# 🔧 DeepHermes 3 — QLoRA Fine-Tune on Colab T4

**Model:** `NousResearch/DeepHermes-3-Llama-3-8B-Preview`  
**Method:** QLoRA (4-bit NF4) via Unsloth  
**GPU:** T4 (16 GB VRAM) — Colab Free Tier

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghassan-gaidi/finetune-lab/blob/main/notebooks/lora/example_qlora_colab.ipynb)

---
## What This Does

1. Loads DeepHermes 3 8B in 4-bit — fits in ~5.5 GB VRAM
2. Adds LoRA adapters on all attention + feed-forward layers
3. Trains on your dataset (conversation format with `<think>` tags)
4. Exports to GGUF `q4_k_m` and pushes to Hugging Face Hub

Requires ~12 GB free disk in Colab runtime for the base model + outputs.

---
## Setup

### 1a. HF Token (recommended: Colab Secrets)

Set your Hugging Face token as a **Colab Secret** instead of hardcoding:
- Left sidebar → 🔑 Secrets → `HF_TOKEN`
- If you skip this, hardcode below (not recommended for shared notebooks).

In [ ]:
import os
import torch

# Try Colab Secrets first, fall back to env var
from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("✅ HF token loaded from Colab Secrets")
except Exception:
    pass

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or hardcode below"
os.environ["WANDB_DISABLED"] = "true"  # no login prompt
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

### 1b. Install Unsloth (GPU-capability aware)

In [ ]:
import torch

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"GPU: {torch.cuda.get_device_name()}  |  Capability: {major}.{minor}")
else:
    raise RuntimeError("No GPU detected — go to Runtime → Change runtime type → T4 GPU")

if major >= 8:
    # Ampere+ (RTX 30xx/40xx, A100, H100)
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
else:
    # Tesla T4, V100 (major=7)
    !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Unsloth pins deps internally — --no-deps avoids version conflicts
!pip install --no-deps "xformers<0.0.27" trl peft transformers accelerate

print("✅ Dependencies installed")

---
## Load Model in 4-bit

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048       # fits T4 comfortably
dtype = torch.float16       # T4 has no native bfloat16
load_in_4bit = True         # 8B model drops from 16GB → ~5.5 GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="NousResearch/DeepHermes-3-Llama-3-8B-Preview",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=os.environ["HF_TOKEN"],           # auth for gated models
)

# Apply LoRA adapters on all linear projection layers
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # prevents OOM
    random_state=3407,
)

print(f"✅ Model loaded. Trainable params: {model.num_parameters(only_trainable=True):,}")

---
## Dataset

### Expected Format

Your dataset must use a **ShareGPT-style conversation** format with a
`"conversations"` column. Each entry:

```json
{
  "conversations": [
    {"from": "human", "value": "What is 2+2?"},
    {"from": "gpt", "value": "<think>The user asked a simple arithmetic question. Apply basic addition.</think>\nThe answer is 4."}
  ]
}
```

DeepHermes-3 is trained to output `<think>...</think>` reasoning blocks.
Make sure your assistant responses include that tag — the `llama-3` chat
template will format it correctly.

> **Replace the dataset path below** with your actual dataset on HF Hub.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Apply Llama 3 chat template with ShareGPT field mapping
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3",
    mapping={"role": "from", "content": "value",
             "user": "human", "assistant": "gpt"},
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

# 🔁 REPLACE THIS with your actual dataset
DATASET_PATH = "your_hf_username/your_agent_dataset"

dataset = load_dataset(DATASET_PATH, split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

print(f"✅ Dataset loaded: {len(dataset)} examples")
print(f"    Example:\n{dataset[0]['text'][:300]}...")

---
## Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,         # small batch = low VRAM
        gradient_accumulation_steps=4,          # effective batch = 8
        warmup_steps=5,
        max_steps=60,                            # switch to num_train_epochs=1 for full pass
        learning_rate=2e-4,
        fp16=True,                                # T4 native math
        bf16=False,
        logging_steps=1,
        optim="adamw_8bit",                      # saves ~2 GB VRAM vs 32-bit Adam
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",                         # no wandb/tensorboard login
        save_strategy="steps",
        save_steps=20,
    ),
)

trainer_stats = trainer.train()
print(f"✅ Training complete — {trainer_stats.metrics['train_loss']:.4f} loss")

---
## Export & Push to HF Hub

Merges LoRA weights into the base model, converts to GGUF `q4_k_m`, and
uploads directly to your Hugging Face repo.

In [ ]:
# 🔁 REPLACE with your target repo
HF_REPO_PATH = "leofalco/deep-hermes-Le0Fvlc0"

print(f"🚀 Merging adapters, converting to GGUF q4_k_m, pushing to {HF_REPO_PATH}...")

model.push_to_hub_gguf(
    repo_id=HF_REPO_PATH,
    tokenizer=tokenizer,
    quantization_method="q4_k_m",
    token=os.environ["HF_TOKEN"],
)

print(f"✅ Done! Model live at https://huggingface.co/{HF_REPO_PATH}")

---

## Test the Exported Model (in llama.cpp)

After the push completes, you can download the GGUF and run inference:

```bash
pip install llama-cpp-python
wget https://huggingface.co/leofalco/deep-hermes-Le0Fvlc0/resolve/main/deep-hermes-Le0Fvlc0-unsloth.Q4_K_M.gguf

python3 -c """
from llama_cpp import Llama
llm = Llama(model_path="./deep-hermes-Le0Fvlc0-unsloth.Q4_K_M.gguf", n_gpu_layers=-1)
out = llm("<|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\n", max_tokens=128)
print(out['choices'][0]['text'])
"""
```